<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-06-advanced-rag-and-open-source-llms/notebooks/Ophelie_Run_First_OpenSource_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Run Your First Open-Source LLM

**Objective:** Load and run an open-source LLM (Mistral-7B-Instruct, quantized GGUF format) locally inside Colab, with no API key required, and compare the experience to using Gemini via an API key.

**Important:** This notebook requires a GPU runtime. In Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, before running any cells below.

## Step 1: Install required packages and check GPU support

In [ ]:
# Install necessary libraries with CUDA support
!pip install -q PyMuPDF
!pip install -q torch

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 81.8 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4


In [ ]:
# Check CUDA version first
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.5/444.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 130.6 MB/s eta 0:00:00


In [ ]:
# Install LlamaIndex
!pip install -q llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


**Why did we install all those packages?**

- torch: Lets us use the GPU, so the model runs way faster.
- llama-cpp-python: The actual engine that runs the Mistral model on our machine.
- llama-index: The retrieval brains (used in the next module).
- fitz (PyMuPDF): Helps us open and read PDFs (used in the next module).

## Step 2: Load Mistral-7B in GGUF format

We're using Mistral-7B-Instruct in GGUF (quantized) format, downloaded from Hugging Face, and loaded with llama-cpp-python.

In [ ]:
from llama_cpp import Llama
import os

# Define model path
model_path = "/content/mistral.gguf"

# Download Mistral model if not already present
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}

print(f"Model downloaded to {model_path}")

# Verify model file exists
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

--2026-08-11 10:00:52--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 18.65.238.63, 18.65.238.97, 18.65.238.93, ...
Connecting to huggingface.co (huggingface.co)|18.65.238.63|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.2.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.2.Q4_K_M.gguf%22%3B&Expires=1786446052&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NzhhYzY2MmQzYWMxODE3Y2M5MjAxLzg2NWY1ZTQ2ODJkZGRiMjljMmUyMDI3MGIyNDcxYTc1OTBjODNhNDE0YmJmMWQ3MmNmNGMwOGZkZmYyZWVjYTRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRp

In [ ]:
# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,
        n_ctx=2048,
        verbose=True
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

## Step 3: Run a basic query to test

This is just a simple prompt to confirm the model loads and responds correctly. The next module plugs this model into a full RAG pipeline.

In [ ]:
import time

prompt = "What is Retrieval-Augmented Generation (RAG)?"
print(f"Sending prompt: {prompt}")

t0 = time.time()
response = llm(prompt, max_tokens=256, temperature=0.1)
elapsed = time.time() - t0

print("\nResponse:")
print(response["choices"][0]["text"])
print(f"\n(generation time: {elapsed:.2f}s)")

Sending prompt: What is Retrieval-Augmented Generation (RAG)?



llama_print_timings:        load time =    7072.71 ms
llama_print_timings:      sample time =      14.14 ms /   256 runs   (    0.06 ms per token, 18109.79 tokens per second)
llama_print_timings: prompt eval time =    7072.52 ms /    15 tokens (  471.50 ms per token,     2.12 tokens per second)
llama_print_timings:        eval time =  159466.63 ms /   255 runs   (  625.36 ms per token,     1.60 tokens per second)
llama_print_timings:       total time =  166795.61 ms /   270 tokens



Response:


Retrieval-Augmented Generation (RAG) is a method used in natural language processing (NLP) and artificial intelligence (AI) to generate human-like text by combining the power of retrieval and generation models.

Retrieval models are designed to search and extract relevant information from large text corpora. Generation models, on the other hand, are capable of generating new text based on the given input. RAG combines these two models to improve the quality and relevance of the generated text.

In RAG, the generation model is given access to a retrieved text snippet or document, which is used as context or input to generate a more accurate and contextually relevant response. This approach has been shown to improve the performance of language models in various applications, such as question answering, text summarization, and conversational AI.

RAG can be implemented using various techniques, such as retrieval-based prompting, retrieval-based fine-tuning, and retrieval-base

## Observations

- **No API key needed.** Unlike Gemini (which needed userdata.get('GOOGLE_API_KEY') and a Google AI Studio key), Mistral-7B here runs entirely inside the Colab VM once the GGUF file is downloaded, no auth, no quota limits, no external service call.
- **Setup cost is real, though.** The trade-off for "no permission needed" is a ~4.1 GB one-time model download plus a from-source-ish install of llama-cpp-python with CUDA wheels, both of which take real wall-clock time before the first token is generated.
- **GPU matters.** With n_gpu_layers=1 and a T4 runtime, generation is usable; on CPU-only this same prompt would be dramatically slower.
- Real prompt, real response, and real timing captured directly from the cell output above, nothing here is invented.